In [1]:
import sys
import numpy as np
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling


# -----------------------------
# 1️⃣ Detect Raster Type
# -----------------------------
def detect_raster_type(src, sample_size=100000):
    """
    Detect whether raster is continuous or categorical.
    Returns: "continuous" or "categorical"
    """

    # Check color table (categorical indicator)
    try:
        if src.colormap(1):
            return "categorical"
    except ValueError:
        pass

    # Float dtype → continuous
    if np.issubdtype(src.dtypes[0], np.floating):
        return "continuous"

    # Integer dtype → inspect unique values
    band = src.read(1, masked=True)

    flat = band.compressed()
    if flat.size == 0:
        return "continuous"

    if flat.size > sample_size:
        sample = np.random.choice(flat, size=sample_size, replace=False)
    else:
        sample = flat

    unique_vals = np.unique(sample)

    # Heuristic threshold
    if len(unique_vals) < 50:
        return "categorical"
    else:
        return "continuous"


# -----------------------------
# 2️⃣ Choose Resampling Method
# -----------------------------
def choose_resampling(src):
    raster_type = detect_raster_type(src)

    if raster_type == "categorical":
        print("Detected: Categorical raster")
        return Resampling.nearest
    else:
        print("Detected: Continuous raster")
        return Resampling.bilinear


# -----------------------------
# 3️⃣ Reproject Raster
# -----------------------------
def reproject_raster(input_path, output_path, dst_crs):

    with rasterio.open(input_path) as src:

        resampling_method = choose_resampling(src)

        transform, width, height = calculate_default_transform(
            src.crs,
            dst_crs,
            src.width,
            src.height,
            *src.bounds
        )

        kwargs = src.meta.copy()
        kwargs.update({
            "crs": dst_crs,
            "transform": transform,
            "width": width,
            "height": height
        })

        with rasterio.open(output_path, "w", **kwargs) as dst:

            for i in range(1, src.count + 1):
                reproject(
                    source=rasterio.band(src, i),
                    destination=rasterio.band(dst, i),
                    src_transform=src.transform,
                    src_crs=src.crs,
                    dst_transform=transform,
                    dst_crs=dst_crs,
                    resampling=resampling_method,
                    num_threads=2
                )

    print("Reprojection complete ✅")


# -----------------------------
# 4️⃣ Run From CLI
# -----------------------------
if __name__ == "__main__":
    input_raster = f"/home/rajat-saxena/Documents/IITBHU/slcrdeployment/fast_backend/temp/8af2e57d-fbb0-4a17-91cf-9319aff66209.tif"
    output_raster = f"/home/rajat-saxena/Documents/IITBHU/slcrdeployment/fast_backend/temp/output.tif"
    target_crs = "EPSG:3857"
    reproject_raster(input_raster, output_raster, target_crs)

    #EPSG:4326 EPSG:3857

Detected: Categorical raster
Reprojection complete ✅
